# Speeding up Gradient Boosting
Random forest is efficient since each tree of the ensemble can be fitted at the same time independently, thus, it scales efficiently with the number of cores and the number of samples.

Gradient boosting, however, is a *sequential* algorithm. It requires the `n - 1` trees to have been fit to be able to fit the `n`th tree.

Ultimately, the most expensive part is the search for the best split in tree which is a brute-force approach (all possible splits are evaluated and the best is picked). So, to accelerate the algorithm, one could reduce the number of splits to be evaluated.

In [1]:
from fontTools.ttLib.tables.D__e_b_g import table_D__e_b_g
from sklearn.datasets import fetch_california_housing

data, target = fetch_california_housing(return_X_y=True)
target *= 100

In [2]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting = GradientBoostingRegressor(n_estimators=200)
cv_results_gbdt = cross_validate(
    gradient_boosting,
    data,
    target,
    scoring='neg_mean_squared_error',
    n_jobs=2
)

In [3]:
print("Gradient Boosting Decision Tree")
print(
    "Mean absolute error via cross-validation: "
    f"{-cv_results_gbdt['test_score'].mean():.3f} ± "
    f"{cv_results_gbdt['test_score'].std():.3f} k$"
)
print(f"Average fit time: {cv_results_gbdt['fit_time'].mean():.3f} seconds")
print(
    f"Average score time: {cv_results_gbdt['score_time'].mean():.3f} seconds"
)

Gradient Boosting Decision Tree
Mean absolute error via cross-validation: 3994.698 ± 422.144 k$
Average fit time: 4.610 seconds
Average score time: 0.005 seconds


One way to reduce the number of splits, thus accelerating gradient boosting, is to bin the data before boosting. The `KBinsDiscretizer` transformer does this.

In [5]:
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer

discretizer = KBinsDiscretizer(
    n_bins=256, encode="ordinal", strategy="quantile", quantile_method="averaged_inverted_cdf"
)
data_trans = discretizer.fit_transform(data)
data_trans

/Users/michael/Development/Learning/scikit/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/michael/Development/Learning/scikit/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 3 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/michael/Development/Learning/scikit/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 6 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/michael/Development/Learning/scikit/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 7 ar

array([[249.,  39., 231., ...,  83., 161.,  30.],
       [248.,  19., 203., ...,  28., 160.,  30.],
       [242.,  49., 249., ..., 125., 159.,  29.],
       ...,
       [ 17.,  15., 126., ...,  49., 199.,  82.],
       [ 23.,  16., 136., ...,  29., 199.,  77.],
       [ 53.,  14., 130., ...,  93., 198.,  81.]], shape=(20640, 8))

After this transformation, we have at most 256 unique values per feature.

In [6]:
from sklearn.pipeline import make_pipeline

gradient_boosting = make_pipeline(
    discretizer, GradientBoostingRegressor(n_estimators=200)
)
cv_results_gbdt = cross_validate(
    gradient_boosting,
    data,
    target,
    scoring="neg_mean_squared_error",
    n_jobs=2
)

/Users/michael/Development/Learning/scikit/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/michael/Development/Learning/scikit/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 3 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/michael/Development/Learning/scikit/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 6 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/michael/Development/Learning/scikit/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 7 ar

In [7]:
print("Gradient Boosting Decision Tree with KBinsDiscretizer")
print(
    "Mean absolute error via cross-validation: "
    f"{-cv_results_gbdt['test_score'].mean():.3f} ± "
    f"{cv_results_gbdt['test_score'].std():.3f} k$"
)
print(f"Average fit time: {cv_results_gbdt['fit_time'].mean():.3f} seconds")
print(
    f"Average score time: {cv_results_gbdt['score_time'].mean():.3f} seconds"
)

Gradient Boosting Decision Tree with KBinsDiscretizer
Mean absolute error via cross-validation: 4027.273 ± 389.023 k$
Average fit time: 2.983 seconds
Average score time: 0.007 seconds


As can be seen the generalization performance is very similar, but the average time to fit has been reduced. scikit-learn provides `HistGradientBoostingClassifier` and `HistGradientBoostingRegressor` to do this simply. Each feature in the data set is first binned by computing histograms, which are later used to evaluate potential splits.

In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor

histogram_gradient_boosting = HistGradientBoostingRegressor(
    max_iter=200, random_state=0
)
cv_results_hgbdt = cross_validate(
    histogram_gradient_boosting,
    data,
    target,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
)

In [9]:
print("Histogram Gradient Boosting Decision Tree")
print(
    "Mean absolute error via cross-validation: "
    f"{-cv_results_hgbdt['test_score'].mean():.3f} ± "
    f"{cv_results_hgbdt['test_score'].std():.3f} k$"
)
print(f"Average fit time: {cv_results_hgbdt['fit_time'].mean():.3f} seconds")
print(
    f"Average score time: {cv_results_hgbdt['score_time'].mean():.3f} seconds"
)

Histogram Gradient Boosting Decision Tree
Mean absolute error via cross-validation: 43.758 ± 2.694 k$
Average fit time: 0.772 seconds
Average score time: 0.014 seconds
